In [ ]:
import cv2
from ultralytics import YOLO

# 1. モデルのロード
model = YOLO("best.pt")

# ==========================================
# ⚙️ 条件設定
# ==========================================
SOURCE = r"C:\YOLOcamera\Camera01_20260729_132727.mp4"  # または SOURCE = 0

TARGET_LABEL = "person"  # 検出したい対象
AREA = [100, 100, 500, 500]  # 指定エリア [x_min, y_min, x_max, y_max]

# 💡 追加: 信頼度のしきい値（0.0 〜 1.0）
# 0.9以上の確信がある場合のみ検出対象として扱う
CONF_THRESHOLD = 0.9 
# ==========================================

area_x1, area_y1, area_x2, area_y2 = AREA

# 動画の読み込み開始
cap = cv2.VideoCapture(SOURCE)

print("🎥 処理を開始します。停止するには画面上で 'q' キーを押してください。")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("動画の再生が終了したか、読み込めませんでした。")
        break

    # YOLOでフレームごとに分析
    results = model(frame, verbose=False)

    is_ok = False

    # 検出された物体のチェック
    for box in results[0].boxes:
        cls_id = int(box.cls[0].item())
        label_name = model.names[cls_id]
        
        # 💡 追加: 信頼度（Confidence）を取得
        conf = box.conf[0].item()

        # ラベルが一致し、かつ信頼度がしきい値（0.9）以上の場合のみ判定に進む
        if label_name == TARGET_LABEL and conf >= CONF_THRESHOLD:
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            # 物体の中心点
            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2

            # エリア内に入っているか判定
            if area_x1 <= center_x <= area_x2 and area_y1 <= center_y <= area_y2:
                is_ok = True
                break  # 1つでも条件を満たせばOK確定

    # --- 画面描画処理 ---
    annotated_frame = results[0].plot()

    if is_ok:
        color = (0, 255, 0)  # 緑
        status_text = "STATUS: OK"
    else:
        color = (0, 0, 255)  # 赤
        status_text = "STATUS: NG"

    # 指定エリアの四角形を描画
    cv2.rectangle(
        annotated_frame, (area_x1, area_y1), (area_x2, area_y2), color, 3
    )

    # 画面にステータス文字を描画
    cv2.putText(
        annotated_frame,
        f"{status_text} ({TARGET_LABEL} >= {CONF_THRESHOLD})",
        (area_x1, area_y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        color,
        2,
    )

    # 画面にリアルタイム表示
    cv2.imshow("YOLO Area & Confidence Detection", annotated_frame)

    # 'q' キーが押されたら終了
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# 後処理
cap.release()
cv2.destroyAllWindows()
print("処理を終了しました。")